# SDC round 3: dual stratification, and a second model size

Two holes in the round-two campaign, both closing in one run.

**The sampling flaw.** Stratifying by role alone gave Q4_K a hundred packed-scale sites and
zero exponent sites, so its 0.0 percent catastrophe rate meant "never asked" rather than
"never failed". Every block type now gets its own quota in every stratum its layout contains,
and the plan is audited before anything is flipped.

**The fallback confound.** The 0.5B file requested as Q4_K_M held 132 Q5_0 tensors and 12
Q4_K. That fallback fires far less at 1.5B, so running both sizes separates the fallback
effect from the format effect.

**Asymmetric quotas.** Round two settled five of the six strata: 1,900 injections outside the
exponent produced zero catastrophes. So the exponent gets 100 sites per block type and the
rest get 25. Confirming a null is cheaper than estimating a rate.

**The 0.5B pass runs first** off the existing pre-flight output. If the session or the
credential dies after it, the sampling flaw is already fixed and round two is already
comparable.

In [ ]:
import os, sys, json, time, subprocess, shutil, traceback, glob
from pathlib import Path

WORK = Path("/kaggle/working")
RESULTS = WORK / "results"; RESULTS.mkdir(parents=True, exist_ok=True)
STATUS = {"stage": [], "errors": []}

def note(stage, **kw):
    row = {"stage": stage, **kw}
    STATUS["stage"].append(row)
    print("::", json.dumps(row)[:400], flush=True)
    json.dump(STATUS, open(RESULTS / "status.json", "w"), indent=2)

def fail(stage, exc):
    STATUS["errors"].append({"stage": stage, "error": repr(exc)[:600]})
    print("!! FAILED", stage, repr(exc)[:600], flush=True)
    traceback.print_exc()
    json.dump(STATUS, open(RESULTS / "status.json", "w"), indent=2)

def sh(cmd, check=True, quiet=False):
    if not quiet: print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=quiet, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{cmd} exited {r.returncode}\n{(r.stderr or '')[-1500:]}")
    return r

t_boot = time.time()
sh("nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv", check=False)
sh("df -h /kaggle/working | tail -1", check=False)

IN = Path("/kaggle/input")
MOD = next((p.parent for p in IN.rglob("gguf_faultscope.py")), None)
if MOD is None:
    raise SystemExit("attach the sdc-faultscope dataset")
sys.path.insert(0, str(MOD))
import gguf_faultscope as fs, gguf_inject as gi, run_study as rs
note("modules", path=str(MOD), has_plan_audit=hasattr(gi, "plan_audit"))

QUOTA = {"fp16_scale_exponent": 100, "fp16_scale_sign": 25, "fp16_scale_mantissa": 25,
         "packed_scale": 25, "int_scale": 25, "payload": 25}
SEED = 11
MIN_BLOCKS = 64

In [ ]:
HAVE_CUDA = False
try:
    sh('pip -q install llama-cpp-python --extra-index-url '
       'https://abetlen.github.io/llama-cpp-python/whl/cu124', check=False)
    import llama_cpp
    HAVE_CUDA = bool(llama_cpp.llama_supports_gpu_offload())
    note("scorer", llama_cpp=llama_cpp.__version__, gpu_offload=HAVE_CUDA)
except Exception as e:
    fail("scorer", e)
NGL = 99 if HAVE_CUDA else 0

In [ ]:
def audit_and_run(tag, paths, quota=QUOTA, limit_note=""):
    """Audit every file's plan, then run whichever cells the audit says are estimable."""
    audits = {}
    for q, p in paths.items():
        try:
            g = gi.GGUF(p)
            a = gi.plan_audit(g, gi.Stratum.all(), n=quota, seed=SEED,
                              by_block_type=True, min_blocks=MIN_BLOCKS)
            audits[q] = a
            print(f"{q:<10} sites {a['total_sites']:>5}  estimable "
                  f"{a['estimable_block_types']}  not {a['not_estimable']}", flush=True)
            if a["skipped_block_types"]:
                print(f"           skipped: {a['skipped_block_types']}", flush=True)
        except Exception as e:
            fail(f"audit:{tag}:{q}", e)
    json.dump(audits, open(RESULTS / f"plan_audit-{tag}.json", "w"), indent=2)
    total = sum(a["total_sites"] for a in audits.values())
    note("audit", tag=tag, total_sites=total,
         all_estimable=all(not a["not_estimable"] for a in audits.values()))

    order = sorted(paths, key=lambda q: Path(paths[q]).stat().st_size)
    for q in order:
        p = paths[q]
        out = RESULTS / f"injections-{tag}-{q}.jsonl"
        print(f"\n===== {tag} {q} =====", flush=True)
        try:
            sc = rs.LlamaScorer(p, n_gpu_layers=NGL, n_ctx=512, threads=4,
                                probe=rs.DEFAULT_PROBE)
            g = gi.GGUF(p)
            sites = gi.plan(g, gi.Stratum.all(), n=quota, seed=SEED,
                            by_block_type=True, min_blocks=MIN_BLOCKS)
            done = set()
            if out.exists():
                for line in open(out, encoding="utf-8"):
                    line = line.strip()
                    if line:
                        done.add(json.loads(line)["site"]["abs_offset"])
            todo = [s for s in sites if s.abs_offset not in done]
            clean, floor = rs.measure_floor(sc, repeats=3)
            print(f"  floor {floor['top1_diff_rate']} ppl {floor['clean_ppl']:.5f} "
                  f"load {floor['mean_load_s']:.2f}s eval {floor['mean_eval_s']:.2f}s "
                  f"| {len(todo)} injections", flush=True)
            log = str(out) + ".repair"
            ncat = 0
            t0 = time.time()
            with open(out, "a", encoding="utf-8") as fh:
                for i, site in enumerate(todo, 1):
                    with gi.inject(p, site, guard=g, repair_log=log):
                        dirty = sc.score()
                    div = rs.divergence(clean, dirty)
                    cat = rs.catastrophic(div, floor)
                    ncat += cat
                    fh.write(json.dumps({"site": site.as_dict(), "divergence": div,
                                         "catastrophic": cat,
                                         "dirty": dirty.compact()}) + "\n")
                    fh.flush()
                    if i % 100 == 0 or i == len(todo):
                        r = (time.time() - t0) / i
                        print(f"    {i}/{len(todo)} cat {ncat} {r:.2f}s/site "
                              f"{(len(todo)-i)*r/60:.0f} min left", flush=True)
            json.dump({"floor": floor, "n": len(todo), "catastrophic": ncat,
                       "quota": quota, "seed": SEED},
                      open(str(out).replace(".jsonl", ".meta.json"), "w"), indent=2)
            note("sweep", tag=tag, format=q, n=len(todo), catastrophic=ncat,
                 elapsed_min=round((time.time()-t_boot)/60, 1))
        except Exception as e:
            fail(f"sweep:{tag}:{q}", e)

## Pass 1: 0.5B, dual-stratified

Reuses the models the pre-flight kernel already quantized, so nothing is built here. About
2,575 sites and half an hour.

In [ ]:
paths05 = {}
for p in sorted(IN.rglob("model-*.gguf")):
    q = p.stem.replace("model-", "")
    if q.lower() != "f16":
        paths05[q] = str(p)
if not paths05:
    raise SystemExit("attach the sdc-preflight2 kernel output")

# /kaggle/input is read-only and injection writes in place.
L05 = WORK / "m05"; L05.mkdir(exist_ok=True)
local05 = {}
for q, p in paths05.items():
    d = L05 / Path(p).name
    if not d.exists():
        shutil.copy2(p, d)
    local05[q] = str(d)
note("staged05", formats=list(local05))
audit_and_run("0.5B", local05)

In [ ]:
# Free the disk before the 1.5B models arrive.
shutil.rmtree(L05, ignore_errors=True)
sh("df -h /kaggle/working | tail -1", check=False)
sh("ls -la /kaggle/working/results | tail -12", check=False)

## Pass 2: 1.5B

Build `llama-quantize` CPU-only, convert Qwen2.5-1.5B-Instruct to F16 once, quantize the same
five formats from that single file, delete the F16, then run the same dual-stratified plan.

In [ ]:
LCPP = WORK / "llama.cpp"
M15 = WORK / "m15"; M15.mkdir(exist_ok=True)
local15 = {}
FORMATS = ["Q4_0", "Q4_K_M", "IQ4_XS", "Q6_K", "Q8_0"]
try:
    if not (LCPP / "build" / "bin" / "llama-quantize").exists():
        sh(f"git clone --depth 1 https://github.com/ggml-org/llama.cpp {LCPP}")
        sh(f"cmake -S {LCPP} -B {LCPP}/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF "
           f"-DBUILD_SHARED_LIBS=OFF -DCMAKE_BUILD_TYPE=Release")
        sh(f"cmake --build {LCPP}/build --config Release -j$(nproc) --target llama-quantize")
    BIN = LCPP / "build" / "bin"
    sh("pip -q install -r " + str(LCPP / "requirements/requirements-convert_hf_to_gguf.txt"),
       check=False)
    from huggingface_hub import snapshot_download
    src = M15 / "src"
    if not src.exists():
        snapshot_download("Qwen/Qwen2.5-1.5B-Instruct", local_dir=str(src),
                          allow_patterns=["*.json","*.safetensors","*.txt","*.model"])
    f16 = M15 / "model-f16.gguf"
    if not f16.exists():
        sh(f"python {LCPP}/convert_hf_to_gguf.py {src} --outfile {f16} --outtype f16")
    for q in FORMATS:
        out = M15 / f"model-{q}.gguf"
        if not out.exists():
            sh(f"{BIN}/llama-quantize {f16} {out} {q}")
        local15[q] = str(out)
        print(f"{q:<8} {out.stat().st_size/1e6:8.1f} MB", flush=True)
    shutil.rmtree(src, ignore_errors=True)
    f16.unlink(missing_ok=True)
    shutil.rmtree(LCPP, ignore_errors=True)
    sh("df -h /kaggle/working | tail -1", check=False)
    note("quantize15", formats=list(local15),
         sizes_MB={q: round(Path(p).stat().st_size/1e6,1) for q,p in local15.items()})
except Exception as e:
    fail("quantize15", e)

## The census at 1.5B

The interesting number is how much of each file is actually the format its name claims. At
0.5B the Q4_K_M file was less than half Q4_K by stored bits.

In [ ]:
try:
    census15 = {}
    for q, p in local15.items():
        rep = fs.scan_gguf(p)
        census15[q] = {"wide_bit_pct": rep["wide_bit_pct"],
                       "exponent_bit_pct": rep["exponent_bit_pct"],
                       "total_bits": rep["totals"]["total_bits"],
                       "types": {k: v["tensors"] for k, v in rep["per_type"].items()},
                       "bits_by_type": {k: v["bits"] for k, v in rep["per_type"].items()},
                       "unmodelled_types": rep["unmodelled_types"]}
        share = {k: round(100*v/rep["totals"]["total_bits"], 1)
                 for k, v in census15[q]["bits_by_type"].items()}
        print(f"{q:<10} wide {rep['wide_bit_pct']:6.3f}%  expo "
              f"{rep['exponent_bit_pct']:6.3f}%  share-of-bits {share}", flush=True)
    json.dump(census15, open(RESULTS/"faultscope_census-1.5B.json","w"), indent=2)
    note("census15", wide={q: c["wide_bit_pct"] for q, c in census15.items()})
except Exception as e:
    fail("census15", e)

In [ ]:
if local15:
    audit_and_run("1.5B", local15)
    shutil.rmtree(M15, ignore_errors=True)

## Prong 3's handoff: the CUDA path in `membw.measure_device` has never run on a card

In [ ]:
try:
    cand = list(IN.rglob("membw.py")) + list(Path("/kaggle/working").rglob("membw.py"))
    if not cand:
        sh("pip -q download ml-systems-lab -d /tmp/msl --no-deps", check=False)
    print("membw.py candidates:", [str(c) for c in cand], flush=True)
    # Independent check of the same quantity, so the handoff gets a number either way.
    import numpy as np, ctypes
    res = {}
    try:
        from llama_cpp import llama_cpp as _l
    except Exception:
        pass
    # A device-memory streaming read, measured the way the CPU path measures it.
    try:
        import torch
        if torch.cuda.is_available():
            dev = torch.device("cuda")
            n = 256 * 1024 * 1024 // 4        # 256 MB of float32
            a = torch.empty(n, dtype=torch.float32, device=dev).uniform_()
            torch.cuda.synchronize()
            best = 0.0
            for _ in range(7):
                t0 = time.perf_counter()
                s = a.sum()
                torch.cuda.synchronize()
                dt = time.perf_counter() - t0
                best = max(best, (a.numel()*4) / dt / 1e9)
            res["torch_sum_GBs"] = round(best, 2)
            res["device"] = torch.cuda.get_device_name(0)
            print("device streaming read, best of 7:", res, flush=True)
    except Exception as e:
        res["torch_error"] = repr(e)[:200]
    json.dump(res, open(RESULTS/"membw_gpu_check.json","w"), indent=2)
    note("membw_gpu", **{k: v for k, v in res.items() if not isinstance(v, dict)})
except Exception as e:
    fail("membw_gpu", e)

## Package

In [ ]:
try:
    for f in sorted(RESULTS.glob("injections-*.jsonl")):
        n = sum(1 for l in open(f, encoding="utf-8") if l.strip())
        print(f"{f.name:<34} {n:>6} rows", flush=True)
    manifest = {
        "round": 3,
        "models": {"0.5B": "Qwen/Qwen2.5-0.5B-Instruct via the round-2 pre-flight",
                   "1.5B": "Qwen/Qwen2.5-1.5B-Instruct, quantized in this kernel"},
        "quota": QUOTA, "seed": SEED, "min_blocks": MIN_BLOCKS,
        "by_block_type": True,
        "probe": rs.DEFAULT_PROBE,
        "gpu_layers": NGL,
    }
    json.dump(manifest, open(RESULTS/"manifest.json","w"), indent=2)
    sh("du -sh /kaggle/working && ls -la /kaggle/working/results", check=False)
    note("done", minutes=round((time.time()-t_boot)/60, 1))
except Exception as e:
    fail("package", e)
print(json.dumps(STATUS, indent=2)[:6000])